# Fine-tune Qwen3 for n8n workflow diagnosis and repair v0.7 — Kaggle

This is a Kaggle port of the verified v0.7 Colab workflow. **The training recipe, dataset preparation, documentation retrieval, evaluation sets and scoring stay the same.** Only the notebook paths, Kaggle setup and artifact export are platform-specific.

- Base model: Qwen3-1.7B, Apache-2.0
- Training: 4-bit QLoRA, LoRA rank 16 / alpha 32, learning rate 1e-5, 2 epochs
- Context: 2,048 tokens; per-device batch 1; gradient accumulation 8
- Evaluation: original 192 held-out cases + separate 48-case challenge set
- Output: a new adapter ZIP and a full evidence ZIP

**This run does not load, modify or overwrite an existing adapter or your Google Drive files.** It starts from the pinned base model and saves to a unique folder under `/kaggle/working`.

## Before running

1. In Notebook Settings, choose **GPU T4 x2** and turn **Internet On**.
2. Run the cells from top to bottom in this same session.
3. Do not edit the training settings or turn on GGUF export for this run.
4. When finished, save a Quick Save version, then download `adapter.zip` and `run-evidence.zip` from the Output panel.

The notebook deliberately exposes only GPU 0 to Python. That keeps this run on one T4 and preserves the original single-GPU batch/gradient-accumulation semantics rather than silently changing the effective batch size.

## 1. Check the Kaggle GPU

Select GPU T4 x2 in the Kaggle settings before running this cell. The next cell makes GPU 0 the only visible device to keep the training setup equivalent to the original single-T4 run.

In [ ]:
import os
# Kaggle's free GPU option is T4 x2. Use one T4 only so the original
# single-GPU batch size and gradient accumulation remain unchanged.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!nvidia-smi

## 2. Install the training stack

Use the same package versions and install sequence as the Colab v0.7 notebook. Kaggle supplies the CUDA-enabled PyTorch build, so this cell does not replace or upgrade PyTorch itself. Keep Internet enabled while installing.

In [ ]:
# Follow Unsloth's official Qwen3 Colab installation sequence.
%pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer pyyaml pytest
%pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
%pip install --no-deps --upgrade "torchao>=0.16.0"
%pip install transformers==4.57.6
%pip install --no-deps trl==0.22.2

## 3. Settings and recipe

The cell below syncs the exact project branch, loads the same n8n repair recipe and pinned Qwen3 model registry, and creates a unique run name. It never writes into the source dataset.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess, sys, os, json, hashlib, time, platform, importlib.metadata
import yaml

REPO_URL = "https://github.com/ProMBZ/finetune-anywhere.git"
REPO_BRANCH = "project-01-n8n-workflow-repair"
ROOT = Path("/kaggle/working/finetune-anywhere")

# Always align the Kaggle checkout with the exact remote feature branch.
# A plain git pull can leave a stale local checkout or obscure which revision ran.
if (ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(ROOT), "fetch", "--prune", "origin", REPO_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(ROOT), "checkout", "--force", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"],
        check=True,
    )
elif ROOT.exists():
    raise RuntimeError(
        f"{ROOT} exists but is not a Git checkout. Remove that folder in Kaggle "
        "and rerun this setup cell."
    )
else:
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(ROOT)],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(ROOT), "fetch", "--prune", "origin", REPO_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(ROOT), "checkout", "--force", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"],
        check=True,
    )

local_commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True
).strip()
remote_commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", f"origin/{REPO_BRANCH}"], text=True
).strip()
if local_commit != remote_commit:
    raise RuntimeError(
        f"Kaggle checkout is stale: local {local_commit}, remote {remote_commit}. "
        "Restart the runtime and rerun this cell."
    )

test_path = ROOT / "tests" / "test_n8n_workflow_notebook.py"
test_source = test_path.read_text(encoding="utf-8")
if "unique_code_cell_containing" not in test_source:
    raise RuntimeError(
        f"Outdated notebook regression test found at commit {local_commit}. "
        "Expected the cell-independent GGUF guard. The repository was not updated correctly."
    )

os.chdir(ROOT)
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

print("Synced feature branch:", REPO_BRANCH)
print("Verified repository commit:", local_commit)
print("Verified notebook regression test:", test_path)
# Load the selected n8n repair recipe and verified model registry.
RECIPE = "n8n-workflow-repair"
recipe_dir = ROOT / "recipes" / RECIPE
DATA_DIR = recipe_dir / "data"
config = yaml.safe_load((recipe_dir / "config.yaml").read_text(encoding="utf-8"))
registry = yaml.safe_load((ROOT / "configs" / "models.yaml").read_text(encoding="utf-8"))
MODEL_ALIAS = config["model"]
if MODEL_ALIAS not in registry["models"]:
    raise ValueError(f"Unknown model alias in recipe config: {MODEL_ALIAS}")
model_entry = registry["models"][MODEL_ALIAS]
if config.get("recipe") != RECIPE:
    raise ValueError(f"Recipe config mismatch: expected {RECIPE!r}, found {config.get('recipe')!r}")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Recipe dataset directory not found: {DATA_DIR}")

# Use a unique run name so a new attempt never overwrites an earlier run.
RUN_NAME = RECIPE + "-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")

print("Recipe:", RECIPE)
print("Model alias:", MODEL_ALIAS)
print("Base model:", model_entry["id"])
print("Dataset directory:", DATA_DIR)
print("Run name:", RUN_NAME)
print("Repository revision:", local_commit)



## 4. Keep this run separate

Kaggle does not mount Google Drive. Save checkpoints and outputs in Kaggle's persistent `/kaggle/working` area. Every attempt gets a unique folder; an existing run is never reused or overwritten.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

OUTPUT_ROOT = Path('/kaggle/working/finetune-anywhere-runs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

OUTPUT = OUTPUT_ROOT / RUN_NAME
if OUTPUT.exists():
    # Preserve every previous run; never overwrite earlier checkpoints or evidence.
    retry_suffix = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
    OUTPUT = OUTPUT_ROOT / f'{RUN_NAME}-retry-{retry_suffix}'
    print('Existing run directory preserved. Starting in a fresh directory:')
OUTPUT.mkdir(parents=True, exist_ok=False)
print('Output directory:', OUTPUT)

## 5. Load the model

Load the same Qwen3-1.7B base model in four-bit precision. On Kaggle T4 x2, Python can see only GPU 0, a single 16 GB T4.

In [ ]:
# Unsloth must be imported before Transformers, PEFT or TRL.
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

assert torch.cuda.is_available(), 'Enable GPU T4 x2 in Kaggle Settings, then restart and run from the first cell.'
assert torch.cuda.device_count() == 1, f'Expected one visible GPU, found {torch.cuda.device_count()}. Restart the session and rerun the GPU preflight.'
assert torch.cuda.get_device_capability(0)[0] >= 7, 'This recipe requires a Turing-class (compute capability 7.0+) GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_entry['id'],  # The verified model ID comes from the registry.
    max_seq_length=config['max_seq_length'],  # Limit conversation length to fit memory.
    dtype=None,  # Let Unsloth select the hardware-appropriate weight dtype.
    load_in_4bit=True,  # Keep the frozen base weights in 4 bit for QLoRA.
)

## 6. Apply LoRA

Train small adapter matrices while keeping the base frozen. Each setting is explained beside its value.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=config['rank'],  # Rank 16 controls how much the adapter can learn.
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],  # Adapt attention and feed-forward projections.
    lora_alpha=config['alpha'],  # Alpha 32 scales the adapter update relative to its rank.
    lora_dropout=0,  # Zero dropout uses Unsloth's optimized LoRA path.
    bias='none',  # Leave the original bias parameters frozen.
    use_gradient_checkpointing='unsloth',  # Recompute activations to reduce training memory.
    random_state=config['seed'],  # Fix the adapter initialization seed.
    use_rslora=False,  # Use ordinary LoRA scaling for this starter.
    loftq_config=None,  # Use standard adapter initialization without LoftQ.
)
model.print_trainable_parameters()

## 7. Set the chat template

Use Qwen3’s own role delimiters and an empty thinking prefix for direct answers. The same non-thinking template is used for training and held-out inference.

In [ ]:
# Persist non-thinking mode so training, reloads and GGUF share the same template.
prefix = '{% set enable_thinking = false %}\n'
assert isinstance(tokenizer.chat_template, str) and 'enable_thinking' in tokenizer.chat_template
if not tokenizer.chat_template.startswith(prefix):
    tokenizer.chat_template = prefix + tokenizer.chat_template

preview = [{'role': 'user', 'content': 'Please order 3 desk lamps.'},
           {'role': 'assistant', 'content': '{"item":"desk lamps","quantity":3,"priority":"normal"}'}]
print(tokenizer.apply_chat_template(preview, tokenize=False, add_generation_prompt=False, enable_thinking=False))

## 8. Load and check the data

Validate exact token lengths and split separation before training. Render chat text once, without adding special tokens twice.

In [ ]:
from datasets import Dataset
from common import read_rows
from validate_dataset import validate
from validate_n8n_dataset import validate_dataset as validate_n8n_recipe
from prepare_n8n_dataset import prepare_dataset

SOURCE_DATA_DIR = DATA_DIR
DATA_DIR = OUTPUT / 'prepared-dataset-v0.7'
preparation_report = prepare_dataset(SOURCE_DATA_DIR, DATA_DIR, train_per_code=64)
(OUTPUT / 'dataset-preparation.json').write_text(json.dumps(preparation_report, indent=2) + chr(10))
print('Prepared dataset v0.7:', preparation_report)

recipe_validation = validate_n8n_recipe(DATA_DIR)
print('Prepared recipe structural validation:', recipe_validation)

# Confirm the exact source revision and regression test before running pytest.
runtime_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
runtime_test = (ROOT / "tests" / "test_n8n_workflow_notebook.py").read_text(encoding="utf-8")
if "unique_code_cell_containing" not in runtime_test:
    raise RuntimeError(
        f"Stale regression test at {runtime_commit}. Restart Colab and rerun the repository setup cell."
    )
print("Running pre-training tests from commit:", runtime_commit)
print("GGUF regression test is the updated content-based version.")

# Run all project unit tests before spending GPU time.
import pytest
test_result = pytest.main([
    '-q',
    'tests/test_score_n8n_repairs.py',
    'tests/test_validate_n8n_dataset.py',
    'tests/test_n8n_docs_retriever.py',
    'tests/test_prepare_n8n_dataset.py',
    'tests/test_n8n_workflow_notebook.py',
])
if test_result != pytest.ExitCode.OK:
    raise RuntimeError(f'Project tests failed before training: {test_result}')

raw = {}
seen_prompts = set()
for split in ['train', 'valid', 'eval']:
    file = DATA_DIR / f'{split}.jsonl'
    errors, lengths, prompts = validate(file, tokenizer=tokenizer, max_tokens=config['max_seq_length'])
    if errors:
        raise ValueError('\n'.join(errors))
    if seen_prompts & prompts:
        raise ValueError(f'{split}: a prompt appears in another split')
    seen_prompts |= prompts
    raw[split] = read_rows(file)
    print(split, len(raw[split]), 'examples; longest raw conversation:', max(lengths), 'tokens')


# Load the independent challenge evaluation separately. It is never passed to prepare_dataset().
CHALLENGE_EVAL_FILE = SOURCE_DATA_DIR / 'challenge-eval-v0.6.jsonl'
challenge_errors, challenge_lengths, challenge_prompts_raw = validate(
    CHALLENGE_EVAL_FILE, tokenizer=tokenizer, max_tokens=config['max_seq_length']
)
if challenge_errors:
    raise ValueError('\n'.join(challenge_errors))
challenge_rows = read_rows(CHALLENGE_EVAL_FILE)
if len(challenge_rows) != 48:
    raise ValueError(f'Expected 48 challenge evaluation cases, found {len(challenge_rows)}')

def prompt_fingerprint(row):
    prompt = json.loads(row['messages'][1]['content'])
    prompt.pop('case_id', None)
    return json.dumps(prompt, sort_keys=True, ensure_ascii=False, separators=(',', ':'))

source_fingerprints = {
    prompt_fingerprint(row)
    for split_rows in raw.values()
    for row in split_rows
}
challenge_fingerprints = [prompt_fingerprint(row) for row in challenge_rows]
if len(set(challenge_fingerprints)) != 48:
    raise ValueError('Challenge evaluation contains duplicate prompts after removing case IDs.')
if source_fingerprints.intersection(challenge_fingerprints):
    raise ValueError('Challenge evaluation overlaps training, validation or legacy evaluation prompts.')
challenge_codes = [json.loads(row['messages'][-1]['content'])['diagnosis_code'] for row in challenge_rows]
if len(set(challenge_codes)) != 48:
    raise ValueError('Challenge evaluation must contain one case for each of the 48 canonical diagnosis codes.')
raw['challenge_eval'] = challenge_rows
print('Independent challenge evaluation:', len(challenge_rows), 'cases; one per diagnosis code')

# Load the canonical diagnosis taxonomy and provide its definitions to every split.
# Keep the codebook in the system instruction instead of duplicating it in every JSONL row.
codebook = json.loads((DATA_DIR / 'diagnosis-codes.json').read_text(encoding='utf-8'))
# Keep all 48 labels, but use short discriminative definitions to leave room for
# the actual workflow and retrieved documentation in the 2,048-token context.
COMPACT_CODE_DEFINITIONS = {
    'EXPRESSION_PREFIX_MISSING': 'missing leading = to evaluate expression',
    'NODE_REFERENCE_INVALID': 'expression names a missing node; repair only when intended existing source is explicit',
    'PROPERTY_PATH_INVALID': 'requested path absent from supplied data; no evidenced alternative path; request review',
    'EXPRESSION_METHOD_INVALID': 'unsupported method or incorrect casing',
    'EXPRESSION_SYNTAX_INVALID': 'invalid expression syntax or delimiter',
    'EXPRESSION_DATE_FORMAT_INVALID': 'use Luxon DateTime, not native Date',
    'FIELD_MISSING_OR_NULL': 'field genuinely absent/null; use an explicit evidence-based fallback',
    'ARRAY_INDEX_INCORRECT': 'wrong array index for requested item',
    'TYPE_COERCION_REQUIRED': 'local workflow field needs another value type; not an external API schema rejection',
    'DEFAULT_OPERATOR_LOSES_FALSY_VALUE': '|| loses valid 0/false/empty; use ??',
    'BOOLEAN_STRING_COERCION': 'serialized boolean text is truthy as a string; parse according to source encoding',
    'NESTED_FIELD_PATH_MISMATCH': 'value exists at a different supplied nested path; use that evidenced path',
    'ITEM_REFERENCE_AMBIGUOUS': '.item cannot uniquely resolve upstream item',
    'PAIRED_ITEM_METADATA_MISSING': 'new Code node outputs omit pairedItem source links',
    'WRONG_ITEM_SELECTOR': 'first/last selects wrong ordered item',
    'FIRST_ITEM_USED_WHEN_ALL_REQUIRED': 'first() used although all items are required',
    'UPSTREAM_NODE_REFERENCE_WRONG': 'expression names wrong existing node although intended connected source exists',
    'ITEM_LINEAGE_UNAVAILABLE': 'source item lineage unavailable; use a singleton selector only if exactly one item is proven, else review',
    'HTTP_METHOD_MISMATCH': 'HTTP method contradicts endpoint contract',
    'HTTP_BODY_NOT_SENT': 'body configured but sendBody disabled',
    'RESPONSE_FORMAT_MISMATCH': 'response format conflicts with downstream need',
    'TIMEOUT_VALUE_INVALID': 'timeout has invalid type or representation',
    'TIMEOUT_TOO_SHORT': 'node timeout below an explicit endpoint SLA; increase only with that evidence',
    'WAIT_DURATION_MISMATCH': 'wait duration differs from requested delay',
    'CONNECTION_MISSING': 'required source-to-target connection absent',
    'CONNECTION_TARGET_INVALID': 'connection target node is absent',
    'IF_BRANCH_MISROUTED': 'IF true/false output targets are swapped',
    'MERGE_INPUT_MISSING': 'required source is not connected to Merge',
    'NODE_DISCONNECTED': 'downstream node unreachable due to missing edge',
    'OUTPUT_INDEX_INVALID': 'source output index is invalid',
    'RETURN_VALUE_NOT_ARRAY': 'all-items mode returns object, not item array',
    'ITEM_JSON_WRAPPER_MISSING': 'output item lacks required json wrapper',
    'EXECUTION_MODE_MISMATCH': 'Code node mode conflicts with script API/output',
    'VARIABLE_UNDEFINED': 'referenced variable is unavailable/undeclared',
    'PRIMITIVE_OUTPUT_RETURNED': 'returns primitive, not an n8n item object',
    'PAIRED_ITEM_NOT_PRESERVED': 'existing pairedItem references are overwritten or point to wrong source items',
    'RATE_LIMIT_RETRY_POLICY_MISSING': 'confirmed rate limit; bounded retry missing',
    'API_PAYLOAD_FIELD_MISMATCH': 'request key missing/wrong per API contract',
    'API_PAYLOAD_TYPE_MISMATCH': 'external API schema rejects the request body field type',
    'UPSTREAM_SERVICE_UNAVAILABLE': 'evidenced remote 5xx; no workflow-only fix',
    'DNS_OR_CONNECTION_FAILURE': 'network failure without proven parameter fault',
    'REQUEST_TIMEOUT': 'runtime request timeout without explicit SLA evidence; abstain and request review',
    'AUTHENTICATION_REJECTED': 'provider rejects auth; owner must verify credentials',
    'PERMISSION_INSUFFICIENT': 'identity lacks required provider permission/scope',
    'CREDENTIAL_REFERENCE_MISSING': 'credentialed node has no credential reference',
    'OAUTH_CREDENTIAL_EXPIRED': 'provider reports expired OAuth; owner must reauthorize',
    'API_KEY_REVOKED_OR_MISSING': 'key rejected/unavailable; never guess replacement',
    'SECRET_VALUE_REDACTED': 'secret absent/redacted; no safe secret patch'
}
allowed_codes = {
    entry['code']
    for entries in codebook['categories'].values()
    for entry in entries
}
missing_definitions = allowed_codes - set(COMPACT_CODE_DEFINITIONS)
if missing_definitions:
    raise ValueError(f"Missing compact diagnosis definitions: {sorted(missing_definitions)}")
codebook_definitions = []
for category, entries in codebook['categories'].items():
    codebook_definitions.append(
        category + ': ' + '; '.join(
            f"{entry['code']}={COMPACT_CODE_DEFINITIONS[entry['code']]}"
            for entry in entries
        )
    )
CODEBOOK_INSTRUCTION = (
    "Choose exactly one diagnosis_code from the category definitions below. Match the "
    "primary root cause, not a downstream symptom. Do not invent codes or synonyms. "
    "Distinguish absent/null fields from values at another supplied nested path; distinguish "
    "zero-based indexes in the current input array from upstream-node item selectors; distinguish "
    "local field conversion from external API schema rejection; distinguish text boolean encodings "
    "from native booleans; distinguish missing pairedItem metadata from overwritten source indexes; "
    "distinguish ambiguous .item selection from lineage destroyed by aggregation; and distinguish "
    "a timeout below an explicit endpoint SLA from an unexplained runtime timeout. "
    "Use human review and an empty patch when evidence is insufficient or credentials/secrets, "
    "ambiguous or unavailable lineage, network failures, unexplained timeouts or provider outages "
    "require owner action. Definitions: " + " | ".join(codebook_definitions)
)
BASE_SYSTEM_INSTRUCTION = (
    "Diagnose n8n workflow failures using only the supplied workflow, error, input, "
    "expected behavior, and retrieved official documentation. Return exactly one JSON "
    "object with keys category, diagnosis_code, faulty_node, root_cause, patch, "
    "needs_human_review. Choose category and diagnosis_code only from the taxonomy below. "
    "patch must be a minimal JSON Patch array using add, replace, or remove with valid "
    "JSON Pointer paths. When evidence is insufficient, item lineage is ambiguous, an "
    "external outage is responsible, or credentials/secrets are involved, use patch=[] "
    "and needs_human_review=true. Never reveal, invent, copy, or modify secrets. Never "
    "claim a patch was applied or executed. Output valid JSON only."
)
for split_rows in raw.values():
    for row in split_rows:
        # Replace the original system prompt; it already contained the same 48-code list.
        # Appending the codebook there duplicated the taxonomy and wasted context tokens.
        row['messages'][0]['content'] = BASE_SYSTEM_INSTRUCTION + "\n\n" + CODEBOOK_INSTRUCTION
print('Canonical diagnosis codes loaded:', sum(len(x) for x in codebook['categories'].values()))
print('System instruction characters:', len(BASE_SYSTEM_INSTRUCTION))
print('Compact codebook instruction characters:', len(CODEBOOK_INSTRUCTION))
print('Combined system instruction characters:', len(BASE_SYSTEM_INSTRUCTION) + 2 + len(CODEBOOK_INSTRUCTION))

# Build one local BM25 index from the official n8n docs listed in the recipe.
# Apply the same retrieval procedure to train, validation and held-out evaluation.
from n8n_docs_retriever import build_index, augment_messages

DOCS_INDEX, DOCS_REPORT = build_index(
    ROOT / 'recipes' / RECIPE / 'knowledge' / 'sources.yaml',
    OUTPUT / 'knowledge',
)
print('Official docs indexed:', DOCS_REPORT['sources_indexed'], '/', DOCS_REPORT['sources_requested'])
print('Documentation chunks:', DOCS_REPORT['chunks'])
if DOCS_REPORT['sources_indexed'] == 0:
    raise RuntimeError('No official n8n documentation sources could be indexed. Check the Colab network/logs.')

def build_augmented_splits(top_k, excerpt_chars):
    result = {}
    for split, rows in raw.items():
        augmented_rows = []
        for row in rows:
            prompt_messages = augment_messages(
                row['messages'][:-1], DOCS_INDEX,
                top_k=top_k, excerpt_chars=excerpt_chars,
            )
            augmented_rows.append({'messages': prompt_messages + [row['messages'][-1]]})
        result[split] = augmented_rows
    return result

def token_length(messages, add_generation_prompt=False):
    return len(tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=add_generation_prompt,
        enable_thinking=False,
    ))

# Fit retrieved context to the configured window using the real tokenizer.
# Measure complete prompt + target messages without adding a generation prefix after
# the assistant target. Held-out generation prompts are checked separately below.
retrieval_attempts = [(2, 280), (1, 120), (1, 60), (0, 0)]
augmented = None
max_augmented_tokens = None
for retrieval_top_k, excerpt_chars in retrieval_attempts:
    candidate = build_augmented_splits(
        top_k=retrieval_top_k, excerpt_chars=excerpt_chars
    )
    candidate_max = max(
        token_length(row['messages'], add_generation_prompt=False)
        for rows in candidate.values()
        for row in rows
    )
    print(
        f"Retrieved context top_k={retrieval_top_k}, excerpt_chars={excerpt_chars}: "
        f"longest complete conversation={candidate_max} tokens"
    )
    augmented = candidate
    max_augmented_tokens = candidate_max
    if candidate_max <= config['max_seq_length']:
        break

if max_augmented_tokens > config['max_seq_length']:
    raise ValueError(
        f"Even without retrieved documentation, the longest complete conversation is "
        f"{max_augmented_tokens} tokens; limit is {config['max_seq_length']}. "
        "Inspect the longest source prompt/target rather than silently truncating it."
    )
if retrieval_top_k == 0:
    print("No retrieved excerpts fit this run; continuing with the full workflow and codebook.")
elif retrieval_top_k == 1 and excerpt_chars < 280:
    print("Reduced retrieved documentation excerpts to fit the context window.")

# TRL 0.22.2 requires {% generation %} markers for assistant_only_loss.
# The stock Qwen3 chat template doesn't provide those markers. Use the documented
# conversational prompt/completion format so loss is applied only to assistant targets.
def as_prompt_completion(row):
    messages = row['messages']
    if len(messages) != 3 or messages[-1].get('role') != 'assistant':
        raise ValueError('Expected system/user prompt messages plus one assistant target.')
    return {'prompt': messages[:-1], 'completion': [messages[-1]]}

train_data = Dataset.from_list([as_prompt_completion(row) for row in augmented['train']])
valid_data = Dataset.from_list([as_prompt_completion(row) for row in augmented['valid']])
# Keep evaluation references outside the generation prompt.
eval_prompts = [row['messages'][:-1] for row in augmented['eval']]
eval_prompt_lengths = [token_length(messages, add_generation_prompt=True) for messages in eval_prompts]
if max(eval_prompt_lengths) > config['max_seq_length']:
    raise ValueError('A held-out prompt exceeds the model context window.')

print('Augmented train examples:', len(train_data))
print('Augmented validation examples:', len(valid_data))
print('Held-out prompts:', len(eval_prompts))
print('Longest augmented conversation:', max_augmented_tokens, 'tokens')

challenge_eval_prompts = [row['messages'][:-1] for row in augmented['challenge_eval']]
challenge_eval_references = [row['messages'][-1]['content'] for row in augmented['challenge_eval']]
challenge_eval_prompt_lengths = [
    token_length(messages, add_generation_prompt=True)
    for messages in challenge_eval_prompts
]
if max(challenge_eval_prompt_lengths) > config['max_seq_length']:
    raise ValueError('A challenge evaluation prompt exceeds the model context window.')

print('Longest evaluation generation prompt:', max(eval_prompt_lengths), 'tokens')
print('Longest challenge prompt:', max(challenge_eval_prompt_lengths), 'tokens')
print('Documentation failures:', DOCS_REPORT['failed_sources'])
preview_messages = train_data[0]['prompt'] + train_data[0]['completion']
print(tokenizer.apply_chat_template(
    preview_messages, tokenize=False,
    add_generation_prompt=False, enable_thinking=False,
))




## 9. Record the base answers

Disable the adapter and run all held-out prompts before training. Keep the final assistant targets out of the input.

In [ ]:
GENERATION = {'max_new_tokens': 384, 'do_sample': False}

def answer(messages):
    text = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors='pt', add_special_tokens=False).to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **inputs, **GENERATION, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        generated[0, inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    ).strip()

FastLanguageModel.for_inference(model)
base_outputs = []
with model.disable_adapter():
    for messages in eval_prompts:
        base_outputs.append(answer(messages))
print('First held-out prompt:', eval_prompts[0][-1]['content'])
print('BASE:', base_outputs[0])
with model.disable_adapter():
    challenge_base_outputs = [answer(messages) for messages in challenge_eval_prompts]
print('Base challenge evaluation cases:', len(challenge_base_outputs))
FastLanguageModel.for_training(model)

## 10. Train

Train on assistant diagnosis/repair completions only, using TRL's conversational prompt/completion format. Inspect validation loss after each epoch; the final evaluation split is never passed to the trainer.

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=valid_data,
    args=SFTConfig(
        output_dir=str(OUTPUT / 'checkpoints'),
        completion_only_loss=True,
        max_length=config['max_seq_length'],
        per_device_train_batch_size=config['batch_size'],
        gradient_accumulation_steps=config['gradient_accumulation_steps'],
        learning_rate=config['learning_rate'],
        num_train_epochs=config['epochs'],
        warmup_ratio=0.06,
        weight_decay=0.01,
        max_grad_norm=0.3,
        optim='adamw_8bit',
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        per_device_eval_batch_size=1,
        eval_strategy='steps',
        eval_steps=128,
        save_strategy='steps',
        save_steps=128,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        logging_steps=10,
        seed=config['seed'],
        report_to='none',
        packing=False,
        padding_free=False,
    ),
)
FastLanguageModel.for_training(model)
torch.cuda.reset_peak_memory_stats()
training_start = time.monotonic()
training_result = trainer.train()
training_seconds = time.monotonic() - training_start
print('Training seconds:', round(training_seconds, 2))
print('Best checkpoint:', trainer.state.best_model_checkpoint)
print('Best validation loss:', trainer.state.best_metric)
print('Peak allocated VRAM (GiB):', round(torch.cuda.max_memory_allocated() / 2**30, 2))

## 11. Test one prompt

Generate the same held-out prompt with the trained adapter and print it beside the actual base answer.

In [ ]:
FastLanguageModel.for_inference(model)
test_messages = eval_prompts[0]
first_tuned_output = answer(test_messages)
print('PROMPT:', test_messages[-1]['content'])
print('BASE:', base_outputs[0])
print('FINE-TUNED:', first_tuned_output)
print('REFERENCE:', raw['eval'][0]['messages'][-1]['content'])

## 12. Evaluate all held-out prompts

Keep every result, including failures. The default recipe also reports exact parsed-JSON matches.

In [ ]:
from IPython.display import display, Markdown
from common import cell

tuned_outputs = [first_tuned_output] + [answer(messages) for messages in eval_prompts[1:]]
challenge_tuned_outputs = [answer(messages) for messages in challenge_eval_prompts]
challenge_records = [
    {'messages': messages, 'reference': reference,
     'base_output': before, 'finetuned_output': after}
    for messages, reference, before, after in zip(
        challenge_eval_prompts, challenge_eval_references,
        challenge_base_outputs, challenge_tuned_outputs
    )
]
challenge_lines = ['# Independent challenge evaluation', '',
    '48 new held-out synthetic cases, one per canonical diagnosis code; not used for training or validation.', '',
    '| Prompt | Reference | Base | Fine-tuned |', '| --- | --- | --- | --- |']
for row in challenge_records:
    challenge_lines.append('| ' + ' | '.join(cell(value) for value in [
        row['messages'][-1]['content'], row['reference'],
        row['base_output'], row['finetuned_output']
    ]) + ' |')
challenge_comparison_md = '\n'.join(challenge_lines) + '\n'

records = [{'messages': messages, 'reference': row['messages'][-1]['content'],
            'base_output': before, 'finetuned_output': after}
           for row, messages, before, after in zip(raw['eval'], eval_prompts, base_outputs, tuned_outputs)]
lines = ['# Held-out comparison', '',
         'Same retrieved official-doc context, same 4-bit base, same prompts and generation settings. Comparison is before GGUF export.', '',
         '| Prompt | Reference | Base | Fine-tuned |', '| --- | --- | --- | --- |']
for row in records:
    lines.append('| ' + ' | '.join(cell(value) for value in [row['messages'][-1]['content'],
                 row['reference'], row['base_output'], row['finetuned_output']]) + ' |')
comparison_md = '\n'.join(lines) + '\n'
# Display a compact preview; keep all 192 full outputs in comparison.json/comparison.md.
preview_lines = lines[:18]
preview_lines.append(f"\nShowing the first 12 of {len(records)} held-out cases. Full results are saved in comparison.json and comparison.md.")
display(Markdown('\n'.join(preview_lines)))

if RECIPE == 'json-extractor':
    def matches(output, reference):
        try:
            parsed = json.loads(output)
            return (isinstance(parsed, dict) and set(parsed) == {'item', 'quantity', 'priority'}
                    and (parsed['item'] is None or isinstance(parsed['item'], str))
                    and (parsed['quantity'] is None or type(parsed['quantity']) is int)
                    and parsed == json.loads(reference))
        except (ValueError, TypeError):
            return False
    for name in ['base_output', 'finetuned_output']:
        print(name, 'exact JSON matches:', sum(matches(row[name], row['reference']) for row in records), '/', len(records))

## 13. Score n8n repair quality

Score the base and fine-tuned model on the same untouched evaluation cases. This checks JSON structure, diagnosis labels, expected patch match, safe abstention and whether the patch can be applied to the supplied synthetic workflow fixture. Patch applicability is **not** proof of execution in a live n8n instance.

In [ ]:
from score_n8n_repairs import score_outputs
from IPython.display import display, Markdown

references = [row['messages'][-1]['content'] for row in raw['eval']]
prompts = [messages[-1]['content'] for messages in eval_prompts]
base_scores = score_outputs(base_outputs, references, prompts)
tuned_scores = score_outputs(tuned_outputs, references, prompts)

def show_scores(label, scores):
    rows = [f"| {key.replace('_', ' ').title()} | {value} |" for key, value in scores.items()
            if isinstance(value, str) and key != 'interpretation']
    display(Markdown(f"### {label}\n\n| Metric | Result |\n|---|---:|\n" + "\n".join(rows)))
    print(scores['interpretation'])

show_scores('Base Qwen3-1.7B', base_scores)
show_scores('Fine-tuned Qwen3-1.7B', tuned_scores)
print('Per-category fine-tuned results:', tuned_scores['by_category'])
print('Per-diagnosis-code results:', tuned_scores['by_diagnosis_code'])
print('Diagnosis-code confusion matrix:', tuned_scores['diagnosis_code_confusion'])
print('Unexpected diagnosis codes:', tuned_scores['unexpected_diagnosis_codes'])

challenge_prompts_for_scoring = [messages[-1]['content'] for messages in challenge_eval_prompts]
challenge_base_scores = score_outputs(
    challenge_base_outputs, challenge_eval_references, challenge_prompts_for_scoring
)
challenge_tuned_scores = score_outputs(
    challenge_tuned_outputs, challenge_eval_references, challenge_prompts_for_scoring
)
show_scores('Independent challenge base Qwen3-1.7B', challenge_base_scores)
show_scores('Independent challenge fine-tuned Qwen3-1.7B', challenge_tuned_scores)
print('Challenge per-category results:', challenge_tuned_scores['by_category'])
print('Challenge per-diagnosis-code results:', challenge_tuned_scores['by_diagnosis_code'])


## 14. Save adapter and evidence

Save before export so the training evidence is already on disk.

In [ ]:
# Keep adapters and evidence before attempting the larger GGUF export.
model.save_pretrained(str(OUTPUT / 'adapter'))
tokenizer.save_pretrained(str(OUTPUT / 'adapter'))

report = {
    'recipe': RECIPE,
    'model_alias': MODEL_ALIAS,
    'model_id': model_entry['id'],
    'loaded_base_revision': getattr(model.config, '_commit_hash', None),
    'catalog_revision': model_entry['revision'],
    'repository_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'gpu': torch.cuda.get_device_name(0),
    'gpu_bytes': torch.cuda.get_device_properties(0).total_memory,
    'python': platform.python_version(),
    'package_versions': {
        name: importlib.metadata.version(name)
        for name in ['unsloth', 'unsloth_zoo', 'torch', 'transformers', 'trl', 'peft', 'datasets']
    },
    'settings': config,
    'enable_thinking': False,
    'generation': GENERATION,
    'assistant_only_loss': False,
    'completion_only_loss': True,
    'training_dataset_format': 'conversational_prompt_completion',
    'dataset_preparation': preparation_report,
    'training_data_version': preparation_report['version'],
    'training_target_per_diagnosis_code': preparation_report['train_per_diagnosis_code'],
    'best_checkpoint': trainer.state.best_model_checkpoint,
    'best_validation_loss': trainer.state.best_metric,
    'challenge_eval_cases': len(challenge_records),
    'challenge_eval_base_scores': challenge_base_scores,
    'challenge_eval_finetuned_scores': challenge_tuned_scores,
    'challenge_eval_sha256': hashlib.sha256(
        (SOURCE_DATA_DIR / 'challenge-eval-v0.6.jsonl').read_bytes()
    ).hexdigest(),
    'supplemental_train_sha256': {
        name: hashlib.sha256((SOURCE_DATA_DIR / name).read_bytes()).hexdigest()
        for name in [
            'supplemental-train-v0.5.jsonl',
            'supplemental-train-v0.6-mapping.jsonl',
            'supplemental-train-v0.6-linking.jsonl',
            'supplemental-train-v0.6-node-config.jsonl',
            'supplemental-train-v0.6-code.jsonl',
            'supplemental-train-v0.6-api.jsonl',
            'supplemental-train-v0.6-expression.jsonl',
            'supplemental-train-v0.6-workflow.jsonl',
            'supplemental-train-v0.6-credentials.jsonl',
            'supplemental-train-v0.7-mapping.jsonl',
            'supplemental-train-v0.7-linking.jsonl',
            'supplemental-train-v0.7-node-config.jsonl',
            'supplemental-train-v0.7-code.jsonl',
            'supplemental-train-v0.7-api.jsonl',
            'supplemental-train-v0.7-expression.jsonl',
            'supplemental-train-v0.7-workflow.jsonl',
            'supplemental-train-v0.7-credentials.jsonl',
        ]
    },
    'supplemental_valid_sha256': {
        name: hashlib.sha256((SOURCE_DATA_DIR / name).read_bytes()).hexdigest()
        for name in ['supplemental-valid-v0.6.jsonl', 'supplemental-valid-v0.7.jsonl']
    },
    'retrieval_enabled': True,
    'documentation_index': DOCS_REPORT,
    'training_seconds': training_seconds,
    'train_metrics': training_result.metrics,
    'peak_allocated_gpu_bytes': torch.cuda.max_memory_allocated(),
    'source_data_sha256': {
        split: hashlib.sha256((SOURCE_DATA_DIR / f'{split}.jsonl').read_bytes()).hexdigest()
        for split in ['train', 'valid', 'eval']
    },
    'prepared_data_sha256': {
        split: hashlib.sha256((DATA_DIR / f'{split}.jsonl').read_bytes()).hexdigest()
        for split in ['train', 'valid', 'eval']
    },
    'split_sizes': {split: len(rows) for split, rows in raw.items()},
    'export_completed': False,
}
(OUTPUT / 'run.json').write_text(json.dumps(report, indent=2) + '\n')
(OUTPUT / 'comparison.json').write_text(json.dumps(records, indent=2, ensure_ascii=False) + '\n')
(OUTPUT / 'comparison.md').write_text(comparison_md)
(OUTPUT / 'challenge-comparison.json').write_text(
    json.dumps(challenge_records, indent=2, ensure_ascii=False) + '\n'
)
(OUTPUT / 'challenge-comparison.md').write_text(challenge_comparison_md)
(OUTPUT / 'challenge-scores.json').write_text(
    json.dumps({'base': challenge_base_scores, 'finetuned': challenge_tuned_scores}, indent=2) + '\n'
)
(OUTPUT / 'training-log.json').write_text(json.dumps(trainer.state.log_history, indent=2) + '\n')
print('Saved adapter and all evaluation evidence:', OUTPUT)


## 15. Export GGUF

Merge the adapter and quantize it to Q4_K_M for Ollama or llama.cpp. Keep the cell output if conversion fails.

In [ ]:
# GGUF export is optional. Leave disabled for this training/evaluation iteration.
EXPORT_GGUF = False

if EXPORT_GGUF:
    export_start = time.monotonic()
    model.save_pretrained_gguf(
        str(OUTPUT / 'gguf'), tokenizer,
        quantization_method='q4_k_m',
    )
    gguf_files = sorted(
        path for path in OUTPUT.rglob('*.gguf')
        if path.is_file() and path.stat().st_size > 0
        and 'Q4_K_M' in path.name.upper()
    )
    assert gguf_files, f'No nonempty Q4_K_M GGUF found under {OUTPUT}.'
    report['export_completed'] = True
    report['export_seconds'] = time.monotonic() - export_start
    report['gguf_files'] = [
        {'name': f.name, 'path': str(f.relative_to(OUTPUT)), 'bytes': f.stat().st_size}
        for f in gguf_files
    ]
    (OUTPUT / 'run.json').write_text(json.dumps(report, indent=2) + '\n')
    for file in gguf_files:
        print(file, round(file.stat().st_size / 1e9, 2), 'GB')
else:
    print('Skipping GGUF export. Adapter and evaluation evidence are saved separately.')

## 16. Save and retrieve the artifacts

The next cell creates two separate download files in `/kaggle/working`:

- `run-evidence.zip`: base-vs-tuned outputs, both evaluation score reports, training log, run metadata and dataset manifest.
- `adapter.zip`: the trained LoRA adapter and tokenizer files. This is the model artifact; it is kept separate from the evidence.

After the cell completes, use the links shown below or open the Kaggle **Output** panel to download both ZIP files. Do not upload model weights to GitHub.

In [ ]:
import zipfile
import shutil
from IPython.display import display, FileLink

bundle = OUTPUT / 'run-evidence.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in [
        'run.json', 'comparison.json', 'comparison.md',
        'challenge-comparison.json', 'challenge-comparison.md',
        'challenge-scores.json', 'training-log.json',
        'dataset-preparation.json'
    ]:
        archive.write(OUTPUT / name, arcname=name)
    docs_manifest = OUTPUT / 'knowledge' / 'manifest.json'
    if docs_manifest.exists():
        archive.write(docs_manifest, arcname='knowledge/manifest.json')

# Package the adapter separately. No existing model files are read or overwritten.
adapter_zip = OUTPUT / 'adapter.zip'
shutil.make_archive(
    str(adapter_zip.with_suffix('')), 'zip',
    root_dir=OUTPUT, base_dir='adapter'
)
assert adapter_zip.is_file() and adapter_zip.stat().st_size > 0, 'Adapter ZIP was not created.'
print('Evidence ZIP:', bundle)
print('Adapter ZIP:', adapter_zip)
print('Full run directory:', OUTPUT)
print('Download both files from the links below or Kaggle Notebook > Output.')
display(FileLink(str(bundle)))
display(FileLink(str(adapter_zip)))

## 17. Final checks

Do not judge the run from training loss alone. Review:

1. The original 192-case held-out metrics.
2. The separate 48-case challenge metrics.
3. Per-category and per-diagnosis-code results, especially data mapping and item linking.
4. Whether the adapter improves patch applicability and safe abstention without creating unsafe fixes.

A better-looking score on one split is not enough. Keep the adapter only after reviewing the full evidence. GGUF export remains disabled.